# 5장 (실습 3): Agent 실행 환경의 Egress를 허용 목록으로 제한해보기

> 책 『AI 보안, 클라우드에서 시작한다』 5장 [실습] 3번을 노트북에서 단계별로 실행하는 버전입니다. 5-4 "Sandbox 격리와 Egress 통제"의 목적지 허용 목록, 기록, 긴급 차단을 Docker로 직접 구성해 봅니다. 소요 시간은 30분입니다.
> 같은 폴더의 [README.md](./README.md)와 내용이 같고, 셀을 위에서부터 차례로 실행하면 됩니다.

## 🎯 학습 목표
1. Agent 컨테이너가 **허용 목록에 있는 목적지로만** 나갈 수 있게 Egress 프록시를 구성합니다.
2. 프록시 설정을 무시해도 나갈 길이 없도록 **내부 네트워크(`internal: true`)** 로 묶어야 하는 이유를 확인합니다.
3. 프록시 기록에서 **거부 기록을 이상 행동의 신호**로 읽고, 출구 하나를 멈춰 **긴급 차단**합니다.

## 📝 실습 구성

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 구성 살펴보기 | 예정 |
| 1 | 실습 환경 띄우기 | 예정 |
| 2 | 허용된 목적지와 허용되지 않은 목적지 | 예정 |
| 3 | 프록시를 우회해 보기 | 예정 |
| 4 | 기록 확인하기 | 예정 |
| 5 | 긴급 차단 해 보기 | 예정 |
| 6 | 이 구성으로 막지 못하는 것 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

## 실습 시나리오

사내 문서를 요약하는 Agent는 업무상 LLM API 하나만 호출하면 됩니다. 그런데 실습 2처럼 문서 속 지시문에 속으면, Agent는 정상 권한으로 다른 곳에 데이터를 보내려 할 수 있습니다. 5-4에서 본 것처럼 들어오는 길을 아무리 잘 막아도 이 경우에는 소용이 없습니다. 그래서 Agent 실행 환경이 나갈 수 있는 곳을 처음부터 허용 목록으로 묶어 둡니다.

> ✅ **실습 전 확인사항**
>
> **1. 실행 환경**<br>
> - 이 노트북은 **리눅스(WSL2의 Ubuntu 포함)에서 Docker를 직접 부를 수 있는 환경**을 가정합니다. 셀 안의 `%%bash`는 그 셀 전체를 bash 명령으로 실행하라는 뜻입니다.
> - Docker Compose와 인터넷 연결이 필요합니다. API Key는 필요 없습니다(키 없이 호출해 서버까지 도달하는지만 봅니다).
>
> **2. 노트북 열기**<br>
> 처음이라면 [SETUP.md](../SETUP.md)의 6단계(VS Code와 Jupyter 노트북 연결)를 먼저 한 번 해 둡니다. 그다음에는 아래 두 방법 가운데 하나로 이 노트북을 엽니다.
> - **VS Code**: Ubuntu 터미널에서 `cd ~/ai-cloud-security-labs && code .`를 실행합니다. 창 왼쪽 아래에 `WSL: Ubuntu-24.04`가 보이면, 왼쪽 탐색기에서 `ch05-agent-mcp-security` 폴더를 펼쳐 이 노트북을 엽니다. 그리고 오른쪽 위 커널 표시를 눌러 **Select Another Kernel...** → **Python Environments...** → `.venv-lab`을 고릅니다. 목록에 `.venv-lab`이 없으면 6단계 ⑥의 2번처럼 `Python: Select Interpreter` → **Enter interpreter path...**로 `~/.venv-lab/bin/python`의 위치를 먼저 알려 줍니다.
> - **웹 브라우저(Jupyter Lab)**: Ubuntu 터미널에서 `cd ~/ai-cloud-security-labs/ch05-agent-mcp-security && ~/.venv-lab/bin/jupyter lab`을 실행하고, 화면에 나온 `http://localhost:8888/...` 주소를 Windows의 브라우저에서 엽니다.
> - 노트북 파일만 따로 내려받거나 다른 폴더로 옮기지 마세요. 노트북은 자기가 들어 있는 `ch05-agent-mcp-security` 폴더의 실습 파일을 읽습니다. 위 두 방법처럼 저장소 안에서 열면 따로 신경 쓸 것은 없습니다.
>
> **3. 기준 버전**<br>
> - 아래 출력은 2026년 10월 23일 Ubuntu 24.04(WSL2), Docker 29.1.3, Docker Compose v2, alpine 3.20의 Squid, curl 8.10.1에서 실행한 결과입니다. 시각, 번호, ID 같은 값은 실행할 때마다 다릅니다. 직접 실행할 때는 출력이 없는 `ch05_lab3_egress_allowlist.ipynb`을 쓰세요.

## 0. 실습 준비와 구성 살펴보기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 구성 살펴보기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 1 | 실습 환경 띄우기 | 예정 |
| 2 | 허용된 목적지와 허용되지 않은 목적지 | 예정 |
| 3 | 프록시를 우회해 보기 | 예정 |
| 4 | 기록 확인하기 | 예정 |
| 5 | 긴급 차단 해 보기 | 예정 |
| 6 | 이 구성으로 막지 못하는 것 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

In [1]:
import json
import os
import shutil
import subprocess
from pathlib import Path

from IPython.display import Markdown, display

LAB = Path.cwd()          # 이 노트북이 있는 폴더
os.environ["PATH"] = f"{Path.home() / 'bin'}:{os.environ['PATH']}"   # 실습 도구를 ~/bin에 둔 경우


def sh(cmd):
    """셸 명령을 실행하고 표준 출력을 문자열로 돌려줍니다."""
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()


def table(header, rows):
    """머리글과 행 목록으로 마크다운 표를 그립니다."""
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    lines += ["| " + " | ".join(str(c) for c in r) + " |" for r in rows]
    display(Markdown("\n".join(lines)))


VERSION_CMD = {
    "docker": "docker --version",
    "docker compose": "docker compose version",
}
for tool, cmd in VERSION_CMD.items():
    state = f"✅ {sh(cmd).splitlines()[0] if sh(cmd) else ''}" if shutil.which(tool) else "❌ 없음"
    print(f"{tool:10} {state}")

docker     ✅ Docker version 29.1.3, build 29.1.3-0ubuntu3~24.04.2
docker compose ❌ 없음


`[표 실습5-4]` Egress 실습 구성 요소

| 구성 요소 | 역할 |
|---|---|
| `agent` 컨테이너 | Agent 역할입니다. curl로 외부 호출을 흉내 냅니다. 내부 네트워크 `agent-net`에만 연결됩니다 |
| `agent-net` 네트워크 | `internal: true`로 만든 내부 네트워크입니다. 여기에는 인터넷으로 나가는 경로가 없습니다 |
| `egress-proxy` 컨테이너 | `agent-net`과 바깥 네트워크에 모두 연결된 유일한 출구입니다. Squid 프록시가 허용 목록의 도메인으로만 연결을 통과시킵니다 |

```text
 ┌──────────── agent-net (internal: true) ────────────┐
 │  agent ──HTTPS_PROXY──▶ egress-proxy (Squid) ──────┼──▶ outside ──▶ 인터넷
 │    └──── 직접 연결: 나갈 경로 없음 ✕               │      (허용 목록의 도메인만)
 └────────────────────────────────────────────────────┘
```

In [2]:
print((LAB / "egress" / "squid.conf").read_text())

# Agent 전용 Egress 프록시: 허용 목록에 있는 도메인으로만 HTTPS 연결을 허용한다
# squid는 http_access 규칙을 위에서부터 차례로 보고, 처음 맞는 규칙을 적용한다.

# 프록시가 요청을 받을 포트 (docker-compose.yml의 HTTPS_PROXY 주소와 같아야 한다)
http_port 3128

# acl = 조건에 이름을 붙이는 문법
# 허용할 도메인. 앞의 점(.)은 하위 도메인까지 포함한다는 뜻 (api.openai.com 등)
acl allowed_sites dstdomain .openai.com .anthropic.com
# HTTPS 포트
acl SSL_ports port 443
# HTTPS 연결은 CONNECT 메서드로 터널을 여는 방식으로 프록시를 지난다
acl CONNECT method CONNECT

# 1) 443이 아닌 포트로 터널을 열려는 요청은 거부 (프록시를 다른 용도로 악용하는 것을 막음)
http_access deny CONNECT !SSL_ports
# 2) 허용 목록 도메인은 통과
http_access allow allowed_sites
# 3) 나머지는 모두 거부 (기본 거부: 허용 목록 방식의 핵심)
http_access deny all

# 누가 어디로 나가려 했는지 기록 (차단된 시도도 남는다 → 이상 징후 확인에 쓴다)
access_log /var/log/squid/access.log
# 실습에서는 캐시가 필요 없으므로 끈다
cache deny all



In [3]:
print((LAB / "egress" / "docker-compose.yml").read_text())

# 5장 실습 3. Agent 실행 환경의 Egress를 허용 목록 프록시로 제한
# 구성: agent 컨테이너는 인터넷과 끊긴 내부망(agent-net)에만 연결되고,
#       밖으로 나가는 길은 egress-proxy 하나뿐이다. 프록시는 squid.conf의 허용 목록만 통과시킨다.
#   agent ──(agent-net, 내부 전용)──▶ egress-proxy ──(outside)──▶ 인터넷(허용 도메인만)
# 실행: docker compose up -d
# 확인 방법은 README의 실습 3 (허용 도메인은 응답이 오고, 그 밖의 도메인은 프록시가 403으로 막는다)
networks:
  agent-net:
    internal: true          # 이 네트워크에서는 인터넷으로 직접 나갈 수 없음
  outside: {}               # 일반 네트워크: 프록시만 이쪽에 연결되어 인터넷으로 나간다

services:
  egress-proxy:
    image: alpine:3.20
    # 컨테이너가 시작될 때 squid를 설치하고 포그라운드(-N)로 실행한다
    command: sh -c "apk add --no-cache squid >/dev/null && squid -N -f /etc/squid/squid.conf"
    volumes:
      - ./squid.conf:/etc/squid/squid.conf:ro   # 허용 목록 설정 파일 (:ro = 읽기 전용으로 연결)
    networks: [agent-net, outside]              # 두 네트워크에 모두 연결된 유일한 컨테이너

  agent:
    image: curlimages/curl:8.10.1   # Agent 역할: curl로 바깥에 요청을 보내 본다
    command: sleep infinity         # 계속 켜 두고 exec로 들어가 명령을 실행한다
    environment:
      H

- `acl allowed_sites dstdomain .openai.com .anthropic.com`: 앞의 점(`.`)은 "이 도메인과 그 아래 모든 주소"라는 뜻입니다. 6단계와 과제 1에서 이 점의 의미를 다시 봅니다.
- `http_access deny all`: 허용 목록에 없으면 모두 거부합니다(기본 거부).
- `agent`에는 `HTTPS_PROXY`만 있고 바깥 네트워크(`outside`)가 연결되어 있지 않습니다.

## 1. 실습 환경 띄우기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 구성 살펴보기 | ✅ 완료 |
| 1 | 실습 환경 띄우기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 2 | 허용된 목적지와 허용되지 않은 목적지 | 예정 |
| 3 | 프록시를 우회해 보기 | 예정 |
| 4 | 기록 확인하기 | 예정 |
| 5 | 긴급 차단 해 보기 | 예정 |
| 6 | 이 구성으로 막지 못하는 것 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

실습 파일을 홈 디렉터리의 `~/egress-lab`으로 복사해 그 안에서 실행합니다. 과제 1에서 `squid.conf`를 고치는데, 저장소의 원본은 그대로 두기 위해서입니다.

> 📌 (참고) 노트북을 다시 실행할 때를 위해, 같은 이름의 실습 환경이 있으면 먼저 내립니다. 처음 실행하면 프록시 컨테이너가 Squid를 설치하느라 10~20초 걸리므로, agent에서 프록시의 3128 포트가 응답할 때까지 기다립니다.

In [4]:
%%bash
[ -d ~/egress-lab ] && (cd ~/egress-lab && docker compose down -t 1 >/dev/null 2>&1)
rm -rf ~/egress-lab && cp -r egress ~/egress-lab
cd ~/egress-lab
docker compose up -d 2>&1 | grep -v '^ *$'
until docker compose exec -T agent curl -s -o /dev/null --max-time 3 http://egress-proxy:3128; do sleep 2; done
docker compose ps --format 'table {{.Service}}\t{{.State}}\t{{.Networks}}'

 Network egress-lab_agent-net  Creating
 Network egress-lab_agent-net  Created
 Network egress-lab_o

utside  Creating
 Network egress-lab_outside  Created
 Container egress-lab-egress-proxy-1  Creating


 Container egress-lab-egress-proxy-1  Created
 Container egress-lab-agent-1  Creating
 Container eg

ress-lab-agent-1  Created
 Container egress-lab-egress-proxy-1  Starting
 Container egress-lab-egres

s-proxy-1  Started
 Container egress-lab-agent-1  Starting
 Container egress-lab-agent-1  Started


두 컨테이너가 어느 네트워크에 붙어 있는지 확인합니다. `agent`는 `agent-net` 하나뿐이어야 합니다.

In [5]:
%%bash
cd ~/egress-lab
for c in $(docker compose ps -q); do
  docker inspect -f '{{.Name}} → {{range $k, $v := .NetworkSettings.Networks}}{{$k}} {{end}}' $c
done
docker network inspect -f '{{.Name}}: internal={{.Internal}}' egress-lab_agent-net egress-lab_outside

/egress-lab-agent-1 → egress-lab_agent-net 


/egress-lab-egress-proxy-1 → egress-lab_agent-net egress-lab_outside 


egress-lab_agent-net: internal=true
egress-lab_outside: internal=false


## 2. 허용된 목적지와 허용되지 않은 목적지

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 구성 살펴보기 | ✅ 완료 |
| 1 | 실습 환경 띄우기 | ✅ 완료 |
| 2 | 허용된 목적지와 허용되지 않은 목적지 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 3 | 프록시를 우회해 보기 | 예정 |
| 4 | 기록 확인하기 | 예정 |
| 5 | 긴급 차단 해 보기 | 예정 |
| 6 | 이 구성으로 막지 못하는 것 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

### 2.1. 시험 함수 만들기

`agent` 컨테이너 안에서 curl로 목적지를 부르고, 결과를 세 가지로 나눕니다.

- **도달**: HTTP 응답 코드를 받았습니다. 401·403·404라도 **목적지 서버가 답한 것**이므로 허용 목록을 통과했습니다.
- **프록시 거부**: 프록시가 연결을 거부했습니다(`CONNECT tunnel failed, response 403`).
- **나갈 길 없음**: 이름을 찾지 못하거나 연결하지 못했습니다.

In [6]:
def probe(url, extra=""):
    """agent 컨테이너에서 url을 부르고 (판정, 자세한 내용)을 돌려줍니다."""
    cmd = (f"cd ~/egress-lab && docker compose exec -T agent curl -sS {extra} --max-time 10 "
           f"-o /dev/null -w '%{{http_code}}' {url}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    code, err = r.stdout.strip(), r.stderr.strip()
    if code and code != "000":
        return f"✅ 도달 (HTTP {code})", "목적지 서버가 응답"
    if "response 403" in err:
        return "⛔ 프록시 거부 (403)", err
    return "⛔ 나갈 길 없음", err


def probe_table(targets, extra=""):
    table(["목적지", "결과", "curl 메시지"], [(t, *probe(t, extra)) for t in targets])

### 2.2. 허용된 목적지로 나가기

README 2단계와 같은 요청입니다.

In [7]:
%%bash
cd ~/egress-lab
docker compose exec -T agent curl -sS -o /dev/null -w "%{http_code}\n" https://api.openai.com/v1/models

401


401은 API Key 없이 호출해서 거절되었다는 뜻입니다. 거절한 쪽이 API 서버이므로, 요청은 허용 목록을 통과해 목적지까지 도달했습니다.

### 2.3. 허용되지 않은 목적지로 나가기

In [8]:
%%bash
cd ~/egress-lab
docker compose exec -T agent curl -sS -o /dev/null -w "%{http_code}\n" https://example.com || true

000


curl: (56) CONNECT tunnel failed, response 403


프록시가 허용 목록에 없는 도메인으로의 연결을 403으로 거부했습니다. Agent가 문서 속 지시를 따라 다른 주소로 무언가를 보내려 해도 같은 결과가 나옵니다.

### 2.4. 여러 목적지를 한 번에 시험하기

공격자가 데이터를 보낼 만한 곳(붙여넣기 사이트, 코드 저장소, 파일 공유)과 업무 목적지를 섞어 시험합니다.

In [9]:
probe_table([
    "https://api.openai.com/v1/models",
    "https://api.anthropic.com/v1/models",
    "https://example.com",
    "https://pastebin.com",
    "https://github.com",
    "https://transfer.sh",
])

| 목적지 | 결과 | curl 메시지 |
|---|---|---|
| https://api.openai.com/v1/models | ✅ 도달 (HTTP 401) | 목적지 서버가 응답 |
| https://api.anthropic.com/v1/models | ✅ 도달 (HTTP 401) | 목적지 서버가 응답 |
| https://example.com | ⛔ 프록시 거부 (403) | curl: (56) CONNECT tunnel failed, response 403 |
| https://pastebin.com | ⛔ 프록시 거부 (403) | curl: (56) CONNECT tunnel failed, response 403 |
| https://github.com | ⛔ 프록시 거부 (403) | curl: (56) CONNECT tunnel failed, response 403 |
| https://transfer.sh | ⛔ 프록시 거부 (403) | curl: (56) CONNECT tunnel failed, response 403 |

## 3. 프록시를 우회해 보기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 구성 살펴보기 | ✅ 완료 |
| 1 | 실습 환경 띄우기 | ✅ 완료 |
| 2 | 허용된 목적지와 허용되지 않은 목적지 | ✅ 완료 |
| 3 | 프록시를 우회해 보기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 4 | 기록 확인하기 | 예정 |
| 5 | 긴급 차단 해 보기 | 예정 |
| 6 | 이 구성으로 막지 못하는 것 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

Agent가 프록시 설정을 무시하고 직접 나가려 하면 어떻게 될까요? `--noproxy '*'`는 모든 주소에 프록시를 쓰지 말라는 옵션입니다.

In [10]:
%%bash
cd ~/egress-lab
docker compose exec -T agent curl -sS --noproxy '*' --max-time 10 -o /dev/null -w "%{http_code}\n" \
  https://api.openai.com/v1/models || true

000


curl: (6) Could not resolve host: api.openai.com


`agent-net`은 내부 네트워크라 바깥 주소의 이름조차 찾을 수 없습니다. 이름 대신 IP 주소로 직접 붙어도 마찬가지입니다.

In [11]:
probe_table(["https://1.1.1.1", "http://example.com"], extra="--noproxy '*'")

| 목적지 | 결과 | curl 메시지 |
|---|---|---|
| https://1.1.1.1 | ⛔ 나갈 길 없음 | curl: (7) Failed to connect to 1.1.1.1 port 443 after 0 ms: Could not connect to server |
| http://example.com | ⛔ 나갈 길 없음 | curl: (6) Could not resolve host: example.com |

허용 목록이 실제로 효과를 내려면 이 조건이 필요합니다. 프록시를 "쓰라고 권하는" 것만으로는 부족하고, 프록시 말고는 **나갈 길이 없어야** 합니다.

> 🙋 (생각해 보기)
> `docker-compose.yml`에서 `agent`의 `networks`에 `outside`를 더하면 어떻게 될까요? 위 두 시험이 모두 "도달"로 바뀝니다. Agent가 프록시 환경변수를 지우는 것만으로 허용 목록을 벗어날 수 있게 됩니다.

## 4. 기록 확인하기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 구성 살펴보기 | ✅ 완료 |
| 1 | 실습 환경 띄우기 | ✅ 완료 |
| 2 | 허용된 목적지와 허용되지 않은 목적지 | ✅ 완료 |
| 3 | 프록시를 우회해 보기 | ✅ 완료 |
| 4 | 기록 확인하기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 5 | 긴급 차단 해 보기 | 예정 |
| 6 | 이 구성으로 막지 못하는 것 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

In [12]:
%%bash
cd ~/egress-lab
docker compose exec -T egress-proxy tail -5 /var/log/squid/access.log

1792684756.877    240 172.20.0.3 TCP_TUNNEL/200 3895 CONNECT api.anthropic.com:443 - HIER_DIRECT/160

.79.104.10 -
1792684757.043      0 172.20.0.3 TCP_DENIED/403 3815 CONNECT example.com:443 - HIER_NON

E/- text/html
1792684757.207      0 172.20.0.3 TCP_DENIED/403 3818 CONNECT pastebin.com:443 - HIER_N

ONE/- text/html
1792684757.374      0 172.20.0.3 TCP_DENIED/403 3812 CONNECT github.com:443 - HIER_N

ONE/- text/html
1792684757.535      0 172.20.0.3 TCP_DENIED/403 3815 CONNECT transfer.sh:443 - HIER_

NONE/- text/html


허용된 연결(`TCP_TUNNEL/200`)과 거부된 연결(`TCP_DENIED/403`)이 모두 남습니다. 3단계의 직접 접속 시도는 프록시를 거치지 않았으므로 **이 기록에는 없습니다**. 그래서 내부 네트워크로 "다른 길"을 아예 없애는 것이 기록의 신뢰성에도 중요합니다.

기록을 파이썬으로 집계해 봅니다. Squid 기록의 한 줄은 `시각 소요시간 출발지IP 결과/코드 크기 방식 목적지 ...` 순서입니다.

In [13]:
def squid_log():
    text = sh("cd ~/egress-lab && docker compose exec -T egress-proxy cat /var/log/squid/access.log")
    rows = []
    for line in text.splitlines():
        f = line.split()
        if len(f) >= 7 and f[5] == "CONNECT":       # HTTPS 연결 요청만 본다
            rows.append({"client": f[2], "result": f[3], "dest": f[6]})
    return rows


from collections import Counter

log = squid_log()
counts = Counter((r["dest"], "거부" if "DENIED" in r["result"] else "허용") for r in log)
table(["목적지", "판정", "횟수"], [(d, v, n) for (d, v), n in sorted(counts.items())])

denied = [r for r in log if "DENIED" in r["result"]]
if denied:
    print(f"🚨 알림: 출발지 {sorted({r['client'] for r in denied})}에서 허용 목록 밖 연결 시도 {len(denied)}건")

| 목적지 | 판정 | 횟수 |
|---|---|---|
| api.anthropic.com:443 | 허용 | 1 |
| api.openai.com:443 | 허용 | 2 |
| example.com:443 | 거부 | 2 |
| github.com:443 | 거부 | 1 |
| pastebin.com:443 | 거부 | 1 |
| transfer.sh:443 | 거부 | 1 |

🚨 알림: 출발지 ['172.20.0.3']에서 허용 목록 밖 연결 시도 5건


업무상 LLM API만 부르면 되는 Agent에서 거부 기록이 나왔다면, 그 자체가 Agent가 평소와 다르게 행동한다는 신호입니다. 5-4의 "Agent 실행 기록과 비정상 행동 탐지"에서처럼, 실무에서는 이 기록을 보안 관제(SIEM)로 보내 알림을 받습니다.

## 5. 긴급 차단 해 보기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 구성 살펴보기 | ✅ 완료 |
| 1 | 실습 환경 띄우기 | ✅ 완료 |
| 2 | 허용된 목적지와 허용되지 않은 목적지 | ✅ 완료 |
| 3 | 프록시를 우회해 보기 | ✅ 완료 |
| 4 | 기록 확인하기 | ✅ 완료 |
| 5 | 긴급 차단 해 보기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 6 | 이 구성으로 막지 못하는 것 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

이상 행동이 확인되면 Agent의 외부 통신을 모두 끊습니다. 이 구성에서는 출구가 하나뿐이므로 그 하나만 멈추면 됩니다.

In [14]:
%%bash
cd ~/egress-lab
docker compose stop egress-proxy
docker compose exec -T agent curl -sS --max-time 10 -o /dev/null -w "%{http_code}\n" https://api.openai.com/v1/models || true

 Container egress-lab-egress-proxy-1  Stopping


 Container egress-lab-egress-proxy-1  Stopped


curl: (5) Could not resolve proxy: egress-proxy


000


허용된 목적지까지 포함해 모든 외부 통신이 끊겼습니다. 조사가 끝나면 다시 엽니다.

> 📌 (참고) `docker compose start egress-proxy`로는 다시 열리지 않습니다. Squid가 멈출 때 남긴 PID 파일(`/var/run/squid.pid`) 때문에 다시 시작한 Squid가 "이미 실행 중"이라며 바로 종료합니다. 그래서 컨테이너를 새로 만들어(`--force-recreate`) 엽니다. 긴급 차단 절차는 **다시 여는 방법까지 미리 시험해 두어야** 한다는 교훈이기도 합니다.

In [15]:
%%bash
cd ~/egress-lab
docker compose up -d --force-recreate egress-proxy 2>&1 | grep -v '^ *$'
until docker compose exec -T agent curl -s -o /dev/null --max-time 3 http://egress-proxy:3128; do sleep 2; done
docker compose exec -T agent curl -sS -o /dev/null -w "다시 열림: %{http_code}\n" https://api.openai.com/v1/models

 Container egress-lab-egress-proxy-1  Recreate
 Container egress-lab-egress-proxy-1  Recreated
 Cont

ainer egress-lab-egress-proxy-1  Starting
 Container egress-lab-egress-proxy-1  Started


`[표 실습5-5]` Egress 통제 결과 정리

| 시도 | 결과 | 막은 장치 |
|---|---|---|
| 허용된 도메인 (api.openai.com) | 401, 서버까지 도달 | 없음 (허용) |
| 허용되지 않은 도메인 (example.com) | 403 거부 | 프록시 허용 목록 |
| 프록시를 우회한 직접 접속 | 이름 확인 실패 | 내부 네트워크 (`internal: true`) |
| 프록시 정지 후 | 프록시 연결 실패 | 긴급 차단 |

## 6. 이 구성으로 막지 못하는 것

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 구성 살펴보기 | ✅ 완료 |
| 1 | 실습 환경 띄우기 | ✅ 완료 |
| 2 | 허용된 목적지와 허용되지 않은 목적지 | ✅ 완료 |
| 3 | 프록시를 우회해 보기 | ✅ 완료 |
| 4 | 기록 확인하기 | ✅ 완료 |
| 5 | 긴급 차단 해 보기 | ✅ 완료 |
| 6 | 이 구성으로 막지 못하는 것 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

- **허용된 목적지로 보내는 유출**: 허용 목록은 "어디로" 가는지를 막을 뿐, 허용된 API로 보내는 요청에 무엇이 담겼는지는 보지 않습니다. 5-4의 콘텐츠 검사와 위험한 행동에 대한 사람 승인을 함께 겹쳐야 합니다.
- **도메인 단위의 허용**: `.openai.com`처럼 도메인 전체를 허용하면 그 아래의 다른 주소도 함께 열립니다. 아래 셀로 확인합니다. 운영에서는 필요한 주소만 좁게 허용하고, AI Gateway(7-4)에서 경로와 API Key 단위로 다시 통제합니다.
- **운영 환경으로의 적용**: 같은 원칙을 쿠버네티스에서는 NetworkPolicy로, 클라우드에서는 VPC의 방화벽 규칙과 Egress 전용 게이트웨이로 구현합니다. 이 실습의 Docker 구성은 원리를 보기 위한 축소판입니다.

In [16]:
probe_table([
    "https://api.openai.com/v1/models",     # 업무에 필요한 주소
    "https://platform.openai.com",          # 같은 도메인 아래의 다른 주소
    "https://cdn.openai.com",
    "https://console.anthropic.com",
])

| 목적지 | 결과 | curl 메시지 |
|---|---|---|
| https://api.openai.com/v1/models | ✅ 도달 (HTTP 401) | 목적지 서버가 응답 |
| https://platform.openai.com | ✅ 도달 (HTTP 403) | 목적지 서버가 응답 |
| https://cdn.openai.com | ✅ 도달 (HTTP 404) | 목적지 서버가 응답 |
| https://console.anthropic.com | ✅ 도달 (HTTP 301) | 목적지 서버가 응답 |

업무에는 `api.openai.com` 하나만 필요한데, 같은 도메인 아래의 다른 주소도 모두 "도달"했습니다. 과제 1에서 허용 목록을 정확한 주소로 좁힙니다.

## 7. 체크포인트와 과제

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 구성 살펴보기 | ✅ 완료 |
| 1 | 실습 환경 띄우기 | ✅ 완료 |
| 2 | 허용된 목적지와 허용되지 않은 목적지 | ✅ 완료 |
| 3 | 프록시를 우회해 보기 | ✅ 완료 |
| 4 | 기록 확인하기 | ✅ 완료 |
| 5 | 긴급 차단 해 보기 | ✅ 완료 |
| 6 | 이 구성으로 막지 못하는 것 | ✅ 완료 |
| 7 | 체크포인트와 과제 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 8 | 정리 | 예정 |

### 7.1. 체크포인트

- [ ] Agent 컨테이너가 허용 목록 밖으로 나가지 못하는 것을 확인했나요?
- [ ] 프록시 설정을 무시해도 나갈 길이 없었던 이유를 설명할 수 있나요?
- [ ] 거부 기록이 왜 이상 행동의 신호가 되는지 이해했나요?
- [ ] 허용된 목적지로 보내는 유출은 이 구성으로 막을 수 없습니다. 실습 2의 결론처럼, 그 위에 어떤 통제를 더 겹쳐야 할까요?

### 7.2. 과제 1: 허용 목록을 정확한 주소로 좁히기

`~/egress-lab/squid.conf`의 `allowed_sites`를 고쳐, 6단계의 네 주소 가운데 **`api.openai.com`만 도달**하게 만듭니다.

1. 아래 셀의 `NEW_ACL`을 고칩니다. 도메인 앞의 점(`.`)을 빼면 "그 주소 하나"만 뜻합니다.
2. 셀을 실행하면 설정 파일을 바꾸고 Squid에 설정을 다시 읽게(`squid -k reconfigure`) 한 뒤 다시 시험합니다.

> 📌 (참고) 설정 파일은 컨테이너에 파일 단위로 마운트되어 있어, `sed -i`처럼 파일을 새로 만드는 방식으로 고치면 컨테이너가 바뀐 내용을 보지 못합니다. 그래서 파이썬으로 같은 파일에 덮어씁니다.

In [17]:
NEW_ACL = "acl allowed_sites dstdomain .openai.com .anthropic.com"   # TODO: 고칩니다

conf = Path.home() / "egress-lab" / "squid.conf"
lines = [NEW_ACL if l.startswith("acl allowed_sites") else l for l in conf.read_text().splitlines()]
conf.write_text("\n".join(lines) + "\n")
print(sh("cd ~/egress-lab && docker compose exec -T egress-proxy squid -k reconfigure 2>&1 && echo 설정 다시 읽음"))
import time; time.sleep(3)

probe_table([
    "https://api.openai.com/v1/models",
    "https://platform.openai.com",
    "https://cdn.openai.com",
    "https://console.anthropic.com",
])

2026/10/22 15:59:48| Processing Configuration File: /etc/squid/squid.conf (depth 0)
2026/10/22 15:59:48| Current Directory is /
설정 다시 읽음


| 목적지 | 결과 | curl 메시지 |
|---|---|---|
| https://api.openai.com/v1/models | ✅ 도달 (HTTP 401) | 목적지 서버가 응답 |
| https://platform.openai.com | ✅ 도달 (HTTP 403) | 목적지 서버가 응답 |
| https://cdn.openai.com | ✅ 도달 (HTTP 404) | 목적지 서버가 응답 |
| https://console.anthropic.com | ✅ 도달 (HTTP 301) | 목적지 서버가 응답 |

<details>
<summary>💡 힌트 보기</summary>

```python
NEW_ACL = "acl allowed_sites dstdomain api.openai.com"
```

Anthropic API도 쓴다면 `api.openai.com api.anthropic.com`처럼 정확한 주소를 나란히 적습니다. 허용 목록은 "도메인"이 아니라 "업무에 필요한 주소" 단위로 관리하고, 주소가 늘어날 때마다 등록·승인 절차를 거칩니다.
</details>

### 7.3. 과제 2: 알림 규칙 정하기 (서술)

4단계에서는 거부 기록이 한 건만 나와도 알림을 냈습니다. 실제 Agent가 수백 개 돌아가는 환경이라면 알림이 너무 많을 수도 있습니다. 아래 상황마다 **즉시 차단(5단계), 담당자 알림, 기록만** 중 무엇을 할지 고르고 이유를 적어 보세요.

| 상황 | 대응 | 이유 |
|---|---|---|
| 요약 Agent가 example.com으로 1회 연결 시도 | | |
| 같은 Agent가 10분 동안 서로 다른 도메인 20곳에 연결 시도 | | |
| 허용된 api.openai.com으로 평소의 50배 크기 요청 | | |

## 8. 정리

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 구성 살펴보기 | ✅ 완료 |
| 1 | 실습 환경 띄우기 | ✅ 완료 |
| 2 | 허용된 목적지와 허용되지 않은 목적지 | ✅ 완료 |
| 3 | 프록시를 우회해 보기 | ✅ 완료 |
| 4 | 기록 확인하기 | ✅ 완료 |
| 5 | 긴급 차단 해 보기 | ✅ 완료 |
| 6 | 이 구성으로 막지 못하는 것 | ✅ 완료 |
| 7 | 체크포인트와 과제 | ✅ 완료 |
| 8 | 정리 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |

In [18]:
%%bash
cd ~/egress-lab && docker compose down -t 1 2>&1 | tail -3
cd ~ && rm -rf ~/egress-lab && echo "실습 폴더 삭제됨"

 Network egress-lab_outside  Removing
 Network egress-lab_outside  Removed
 Network egress-lab_agent

-net  Removed


실습 폴더 삭제됨


---
> 7장 실습 2(미니 제로트러스트 랩)에서는 이 실습의 "출구 하나" 원칙을 사용자 요청 쪽으로 넓혀, 모든 요청이 정책 집행 지점(PEP)을 거치게 만듭니다.